# Test LULC outputs against the held-out test data

Compares LULC classifications with the frozen test points (`TEST_TILES` in `ldn/training_data.py`, written by the
`test-data-generate-*` Makefile targets to `dep_ls_lulc/test_data/{TEST_DATA_VERSION}/`). This is seperate to the 
validation workflow in the README: programmatic validation used to compare data and model versions. That uses data created 
by locals/experts. These test labels come from agreement between at least 2 of ESA WorldCover, ESA CCI and Impact Observatory 
(only WorldCover and IO in 2021), so they are a reference, not ground truth. Each year is scored on its own.

1. Load the test points for every test tile and year in `TRAINING_DATA_YEARS`.
2. Find each tile's LULC classification for the same year and sample it at the test points (points are 30 m pixel centres on the same
   grid, so this is an exact pixel lookup).
3. Score: confusion matrix, overall accuracy, Cohen's kappa, per-class precision (user's accuracy), recall (producer's
   accuracy) and F1, with 95% confidence intervals from a bootstrap that resamples whole tiles.
4. Optionally compare two LULC versions on the same points (exact McNemar test).
5. Write a markdown report and confusion matrix figures.

We should expect LULC v0-0-9 to perform badly in the non-Pacific region because it uses only Pacific training data.

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from dep_tools.namers import GenericItemPath
from rustac import search_sync
from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score, confusion_matrix, precision_recall_fscore_support

from ldn.training_data import TEST_TILES
from ldn.typology import classes_flipped
from ldn.utils import (
    CLASS_ATTR,
    LULC_DATASET_ID,
    LULC_NODATA,
    SENSOR,
    TEST_DATA_VERSION,
    TRAINING_DATA_YEARS,
    WGS84,
    dataset_prefix,
    owner_for_region,
    parse_tile_id,
)

In [ ]:
BUCKET_URL = "https://dep-public-staging.s3.us-west-2.amazonaws.com"
TEST_DATA_OWNER = "dep"  # The test-data Makefile targets write both regions' training data under dep_ls_lulc
# Each year's test labels are compared with that year's LULC.
YEARS = TRAINING_DATA_YEARS
# TODO: Generate 2021 test data for all TEST TILES

LULC_VERSIONS = ["0-2-0"]  # LULC versions in S3. Compare 2. Put newest first to prioritise.
# Previous versions to 0-2-0 are not directly comparable.
NODATA = LULC_NODATA
N_BOOTSTRAP = 1000
SEED = 0
REPORT_DIR = Path("reports_test")  # Relative to this notebook (notebooks/LULC_Testing)

CLASSES = sorted(c for c in classes_flipped if c != NODATA)
LABELS = [classes_flipped[c] for c in CLASSES]

## Load test points

In [ ]:
def load_test_points(tiles: list, version: str, years: list[str]) -> tuple[gpd.GeoDataFrame, list[str]]:
    """Load the test points for each tile and year from S3.

    Args:
        tiles: (tile_id, region, country) tuples, e.g. TEST_TILES.
        version: Test data version, e.g. TEST_DATA_VERSION.
        years: Years of the test labels.

    Returns:
        Points with reference class, tile_id, region and year (EPSG:4326), and the "tile year"s with no test data yet.
    """
    prefix = dataset_prefix(TEST_DATA_OWNER, SENSOR, LULC_DATASET_ID)
    frames, missing = [], []
    for year in years:
        for tile_id, region, _ in tiles:
            x, y = parse_tile_id(tile_id)
            url = f"{BUCKET_URL}/{prefix}/test_data/{version}/{region}/{x:03d}/{y:03d}/{year}/samples.csv"
            try:
                df = pd.read_csv(url, usecols=[CLASS_ATTR, "geometry"])
            except HTTPError:
                missing.append(f"{tile_id} {year}")
                continue
            frames.append(df.assign(tile_id=tile_id, region=region, year=year))
    points = pd.concat(frames, ignore_index=True).rename(columns={CLASS_ATTR: "reference"})
    return gpd.GeoDataFrame(points, geometry=gpd.GeoSeries.from_wkt(points["geometry"]), crs=WGS84), missing


points, missing_test_data = load_test_points(TEST_TILES, TEST_DATA_VERSION, YEARS)
print(f"Loaded {len(points)} test points for {points.groupby(['tile_id', 'year']).ngroups} tile years")
print(f"No test data yet for: {missing_test_data}")
points.groupby(["year", "region", "reference"]).size().unstack(fill_value=0).rename(columns=classes_flipped)

## Sample LULC at the test points

In [ ]:
def lulc_hrefs(tiles: list, version: str, year: str) -> dict[str, str]:
    """Find the LULC classification COG for each tile from the STAC-Geoparquet index of each owner.

    Args:
        tiles: (tile_id, region, country) tuples.
        version: LULC version, e.g. "0-0-9".
        year: LULC year.

    Returns:
        Classification COG URL keyed by tile id, for tiles that have a LULC output.
    """
    hrefs = {}
    for owner in sorted({owner_for_region(region) for _, region, _ in tiles}):
        prefix = dataset_prefix(owner, SENSOR, LULC_DATASET_ID)
        # Same ID construction as the LULC writer (itempath.basename)
        namer = GenericItemPath(sensor=SENSOR, dataset_id=LULC_DATASET_ID, version=version, time=year, prefix=owner)
        ids = {namer.basename(parse_tile_id(t)): t for t, region, _ in tiles if owner_for_region(region) == owner}
        for item in search_sync(f"{BUCKET_URL}/{prefix}/{version}/{prefix}.parquet", ids=list(ids)):
            hrefs[ids[item["id"]]] = item["assets"]["classification"]["href"]
    return hrefs


def sample_lulc(points: gpd.GeoDataFrame, hrefs: dict[str, str]) -> pd.Series:
    """Sample the LULC classification at each point.

    Args:
        points: Test points with a tile_id column.
        hrefs: Classification COG URL keyed by tile id.

    Returns:
        Predicted class per point. NaN where the tile has no LULC output, NODATA where the model was not confident.
    """
    predicted = pd.Series(np.nan, index=points.index)
    for tile_id, tile_points in points.groupby("tile_id"):
        if tile_id not in hrefs:
            continue
        with rasterio.open(hrefs[tile_id]) as src:
            xy = tile_points.to_crs(src.crs).geometry
            predicted[tile_points.index] = [value[0] for value in src.sample(zip(xy.x, xy.y))]
    return predicted


for version in LULC_VERSIONS:
    points[f"predicted_{version}"] = np.nan
    for year, year_points in points.groupby("year"):
        hrefs = lulc_hrefs(TEST_TILES, version, year)
        points.loc[year_points.index, f"predicted_{version}"] = sample_lulc(year_points, hrefs)
        print(f"LULC {version} ({year}): found for {len(hrefs)} of {len(TEST_TILES)} test tiles")

## Coverage

In [ ]:
version = LULC_VERSIONS[0]
predicted_col = f"predicted_{version}"
version_dir = REPORT_DIR / version
version_dir.mkdir(parents=True, exist_ok=True)
predicted = points[predicted_col]
scored = points[predicted.notna() & (predicted != NODATA)].copy()
scored["predicted"] = scored[predicted_col].astype(int)
coverage = (
    pd.DataFrame(
        {
            "test points": points.groupby("year").size(),
            "tile has no LULC": predicted.isna().groupby(points["year"]).sum(),
            "LULC nodata (below probability threshold or no geomad)": (predicted == NODATA)
            .groupby(points["year"])
            .sum(),
            "scored points": scored.groupby("year").size(),
        }
    )
    .fillna(0)
    .astype(int)
)
assert not scored.empty, f"No test points have LULC {version} for {YEARS}. Run LULC for the test tiles first."
coverage

## Accuracy by region, with tile bootstrap confidence intervals

In [ ]:
def accuracy_metrics(reference: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    """Overall accuracy, Cohen's kappa, macro F1, and per-class precision, recall and F1.

    Args:
        reference: Reference classes.
        predicted: Predicted classes.

    Returns:
        Metric name to value. Per-class metrics are NaN when the class is absent.
    """
    precision, recall, f1, _ = precision_recall_fscore_support(
        reference, predicted, labels=CLASSES, zero_division=np.nan
    )
    metrics = {
        "overall_accuracy": float(np.mean(reference == predicted)),
        "kappa": cohen_kappa_score(reference, predicted, labels=CLASSES),
        "macro_f1": float(np.nanmean(f1)),
    }
    for c, p, r, f in zip(CLASSES, precision, recall, f1):
        metrics |= {f"precision_{c}": p, f"recall_{c}": r, f"f1_{c}": f}
    return metrics


def tile_bootstrap(scored: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """95% confidence intervals for each metric by resampling whole tiles with replacement.

    Points within a tile are spatially correlated, so resampling tiles rather than points gives honest intervals.

    Args:
        scored: Points with reference, predicted and tile_id columns.
        n: Number of bootstrap replicates.
        seed: Random seed.

    Returns:
        ci_low and ci_high for each metric.
    """
    rng = np.random.default_rng(seed)
    tiles = [tile for _, tile in scored.groupby("tile_id")]
    replicates = []
    for _ in range(n):
        sample = pd.concat([tiles[i] for i in rng.integers(len(tiles), size=len(tiles))])
        replicates.append(accuracy_metrics(sample["reference"].to_numpy(), sample["predicted"].to_numpy()))
    return pd.DataFrame(replicates).quantile([0.025, 0.975]).T.set_axis(["ci_low", "ci_high"], axis=1)


def per_class_table(metrics: pd.DataFrame, scored: pd.DataFrame) -> pd.DataFrame:
    """Point counts and per-class precision, recall and F1 with 95% confidence intervals.

    Args:
        metrics: Metric values and confidence intervals, from accuracy_metrics and tile_bootstrap.
        scored: The scored points the metrics are for.

    Returns:
        One row per class.
    """
    return pd.DataFrame(
        {
            "class": LABELS,
            "reference points": [int((scored["reference"] == c).sum()) for c in CLASSES],
            "predicted points": [int((scored["predicted"] == c).sum()) for c in CLASSES],
            **{
                f"{name} (95% CI)": [
                    f"{metrics.loc[f'{key}_{c}', 'value']:.2f} ({metrics.loc[f'{key}_{c}', 'ci_low']:.2f}-"
                    f"{metrics.loc[f'{key}_{c}', 'ci_high']:.2f})"
                    for c in CLASSES
                ]
                for key, name in [("precision", "user's accuracy"), ("recall", "producer's accuracy"), ("f1", "F1")]
            },
        }
    )


# For each year, all test tiles together, then each region on its own.
GROUPS = {}
for year, year_scored in scored.groupby("year"):
    GROUPS[(year, "All regions")] = year_scored
    GROUPS |= {(year, region): group for region, group in year_scored.groupby("region")}
summaries, per_class_tables = {}, {}
for (year, name), group in GROUPS.items():
    metrics = pd.Series(accuracy_metrics(group["reference"].to_numpy(), group["predicted"].to_numpy()), name="value")
    metrics = pd.concat([metrics, tile_bootstrap(group, N_BOOTSTRAP, SEED)], axis=1)
    summaries[(year, name)] = metrics.loc[["overall_accuracy", "kappa", "macro_f1"]]
    per_class_tables[(year, name)] = per_class_table(metrics, group)
    print(year, name)
    display(per_class_tables[(year, name)])

summary = pd.concat(summaries, names=["year", "region", "metric"]).round(3)
summary

# TODO: Does this output the same result as the test in the train model notebook? Or is this more detailed?

## Confusion matrix by region

In [ ]:
def plot_confusion(scored: pd.DataFrame, title: str, png: Path):
    """Plot and save a confusion matrix with the count and share of each reference class.

    Args:
        scored: Points with reference and predicted columns.
        title: Plot title.
        png: Path to save the figure to.
    """
    counts = confusion_matrix(scored["reference"], scored["predicted"], labels=CLASSES)
    row_share = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            colour = "white" if row_share[i, j] > 0.5 else "black"
            ax.text(j, i, f"{counts[i, j]}\n{row_share[i, j]:.0%}", ha="center", va="center", fontsize=8, color=colour)
    ax.set_xticks(range(len(CLASSES)), LABELS, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES)), LABELS)
    ax.set_xlabel(f"LULC {version}")
    ax.set_ylabel("Reference")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(png, dpi=150)
    plt.show()


confusion_pngs = {}
for (year, name), group in GROUPS.items():
    slug = "all" if name == "All regions" else name
    confusion_pngs[(year, name)] = version_dir / f"confusion_matrix_{year}_{slug}.png"
    title = f"Confusion matrix, {year}, {name} (count and share of reference class)"
    plot_confusion(group, title, confusion_pngs[(year, name)])

## Accuracy by tile

In [ ]:
# TODO: use F1 score instead of overall accuracy. TODO
def group_accuracy(scored: pd.DataFrame, by: list[str]) -> pd.DataFrame:
    """Point count and overall accuracy per group.

    Args:
        scored: Points with reference and predicted columns.
        by: Columns to group by.

    Returns:
        points and overall_accuracy per group.
    """
    correct = scored["reference"] == scored["predicted"]
    return correct.groupby([scored[c] for c in by]).agg(points="size", overall_accuracy="mean").round(3)


by_tile = group_accuracy(scored, ["year", "region", "tile_id"]).sort_values("overall_accuracy")
by_tile

# v0-2-0 now Guinea-Bissau 193_120 is the worst-performing tile according to overall accuracy (by far) 0.262

## Compare two LULC versions (exact McNemar test)

In [ ]:
def mcnemar_exact(reference: np.ndarray, predicted_a: np.ndarray, predicted_b: np.ndarray) -> dict[str, float]:
    """Exact McNemar test of whether two classifications have different error rates on the same points.

    Only points where exactly one version is correct count. Points within a tile are correlated, so treat the
    p-value as indicative rather than exact.

    Args:
        reference: Reference classes.
        predicted_a: Classes from version A.
        predicted_b: Classes from version B.

    Returns:
        Points only A got right, points only B got right, and the two-sided p-value.
    """
    a_only = int(np.sum((predicted_a == reference) & (predicted_b != reference)))
    b_only = int(np.sum((predicted_b == reference) & (predicted_a != reference)))
    p_value = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return {"only_a_correct": a_only, "only_b_correct": b_only, "p_value": p_value}


comparison = None
if len(LULC_VERSIONS) == 2:
    cols = [f"predicted_{v}" for v in LULC_VERSIONS]
    both = points[points[cols].notna().all(axis=1) & (points[cols] != NODATA).all(axis=1)]
    if both.empty:
        print(
            f"No test points have LULC from both versions {LULC_VERSIONS}, so they cannot be compared. "
            f"Classify the test tiles for {YEARS} with each version first."
        )
    else:
        comparison = mcnemar_exact(both["reference"].to_numpy(), *(both[c].to_numpy() for c in cols))
        comparison |= {
            f"accuracy_{v}": float(np.mean(both[c] == both["reference"])) for v, c in zip(LULC_VERSIONS, cols)
        }
        comparison["points"] = len(both)
comparison

## Write the report

In [ ]:
def markdown_table(df: pd.DataFrame) -> str:
    """Render a DataFrame as a markdown table.

    Args:
        df: Table to render. The index is written as columns.

    Returns:
        Markdown table string.
    """
    df = df.reset_index().rename(columns={"index": ""})
    rows = [list(map(str, df.columns)), ["-"] * len(df.columns), *df.astype(str).values.tolist()]
    return "\n".join(f"| {' | '.join(row)} |" for row in rows)


comparison_section = (
    f"""
## Version comparison (exact McNemar test)

{markdown_table(pd.Series(comparison, name="value").to_frame())}

A small p-value means the two versions have different accuracy on the same points.
"""
    if comparison
    else ""
)
summary_table = summary.rename(
    index={"overall_accuracy": "Overall accuracy", "kappa": "Cohen's kappa", "macro_f1": "Macro F1"}
)
region_sections = "\n".join(
    f"""
### {year}, {name}

{markdown_table(per_class_tables[(year, name)].set_index("class"))}

![Confusion matrix, {year}, {name}]({confusion_pngs[(year, name)].name})
"""
    for year, name in GROUPS
)

report = f"""# LULC test report: version {version}

- Test data: version {TEST_DATA_VERSION}, {len(TEST_TILES)} test tiles
- LULC: version {version}
- Years: {", ".join(YEARS)}. Each year's labels are compared with that year's LULC.
- Tile years with no test data yet: {", ".join(missing_test_data) or "none"}

## Coverage

{markdown_table(coverage)}

## Overall accuracy by year and region

95% confidence intervals from {N_BOOTSTRAP} bootstrap replicates that resample whole tiles (within each region).

{markdown_table(summary_table)}

## Per class and confusion matrix by year and region

User's accuracy is precision (of points mapped as a class, the share that are that class). Producer's accuracy is
recall (of points that are a class, the share mapped as it).
{region_sections}
## By tile (lowest accuracy first)

{markdown_table(by_tile)}
{comparison_section}
## Caveats

- Reference labels come from agreement between at least 2 of ESA WorldCover, ESA CCI and Impact Observatory (only
  WorldCover and IO in 2021), not ground truth. Agreement favours easy, homogeneous areas, so these accuracies are
  likely higher than on the whole map. The products differ by year, so compare years with care.
- Points are sampled with a minimum per class, so overall accuracy is a sample statistic, not an area-weighted map
  accuracy.
- Points within a tile are spatially correlated. The confidence intervals resample tiles to account for this, but with
  few tiles they are wide and approximate.
"""
report_path = version_dir / "test_lulc.md"
report_path.write_text(report)
print(f"Wrote {report_path}")